# Combining SQL + Pandas for Deeper Insights

This notebook is part of issue #15 (Combining SQL + Pandas for Deeper Insights).
It builds on the `.env` credential setup from issue #14: connect to a local PostgreSQL
database, write SQL to fetch data, load the results into Pandas, and do deeper analysis
that would be awkward to express in SQL alone.

The database runs locally in Docker (`postgres:16`) and is seeded below with a small
sample dataset shaped like Focus Bear usage data (users and their daily focus sessions),
self-contained so the notebook doesn't depend on production data.

## Step 0: Connect using the credentials from `.env`

In [1]:
from dotenv import load_dotenv
import os
import psycopg2
import pandas as pd

load_dotenv()

conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
)
print("Connected.")

Connected.


## Step 1: Seed a small sample dataset

Two tables: `users` and `focus_sessions`, similar in shape to what Focus Bear's own usage
data might look like. Recreated each run so the notebook is repeatable.

In [2]:
with conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS focus_sessions;")
    cur.execute("DROP TABLE IF EXISTS users;")
    cur.execute("""
        CREATE TABLE users (
            user_id SERIAL PRIMARY KEY,
            name TEXT NOT NULL,
            plan TEXT NOT NULL
        );
    """)
    cur.execute("""
        CREATE TABLE focus_sessions (
            session_id SERIAL PRIMARY KEY,
            user_id INTEGER REFERENCES users(user_id),
            session_date DATE NOT NULL,
            focus_minutes INTEGER NOT NULL,
            device TEXT NOT NULL
        );
    """)
    cur.execute("""
        INSERT INTO users (name, plan) VALUES
            ('Alice', 'premium'),
            ('Bob', 'free'),
            ('Chloe', 'premium'),
            ('Dev', 'free');
    """)
    cur.execute("""
        INSERT INTO focus_sessions (user_id, session_date, focus_minutes, device) VALUES
            (1, '2026-08-01', 45, 'desktop'),
            (1, '2026-08-02', 60, 'mobile'),
            (1, '2026-08-03', 30, 'desktop'),
            (2, '2026-08-01', 20, 'mobile'),
            (2, '2026-08-02', 25, 'mobile'),
            (3, '2026-08-01', 90, 'desktop'),
            (3, '2026-08-02', 75, 'desktop'),
            (3, '2026-08-03', 80, 'mobile'),
            (4, '2026-08-01', 15, 'mobile'),
            (4, '2026-08-03', 40, 'desktop');
    """)
conn.commit()
print("Seed data inserted.")

Seed data inserted.


## Step 2: Write a SQL query and load it straight into a DataFrame

Joining `users` and `focus_sessions` in SQL, then handing the result to
`pandas.read_sql_query` rather than pulling raw rows and building the frame by hand.

In [3]:
query = """
    SELECT u.user_id, u.name, u.plan, s.session_date, s.focus_minutes, s.device
    FROM focus_sessions s
    JOIN users u ON u.user_id = s.user_id
    ORDER BY u.user_id, s.session_date;
"""

df = pd.read_sql_query(query, conn)
df

C:\Users\koush\AppData\Local\Temp\ipykernel_46972\3724809606.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,user_id,name,plan,session_date,focus_minutes,device
0,1,Alice,premium,2026-08-01,45,desktop
1,1,Alice,premium,2026-08-02,60,mobile
2,1,Alice,premium,2026-08-03,30,desktop
3,2,Bob,free,2026-08-01,20,mobile
4,2,Bob,free,2026-08-02,25,mobile
5,3,Chloe,premium,2026-08-01,90,desktop
6,3,Chloe,premium,2026-08-02,75,desktop
7,3,Chloe,premium,2026-08-03,80,mobile
8,4,Dev,free,2026-08-01,15,mobile
9,4,Dev,free,2026-08-03,40,desktop


## Step 3: Deeper analysis in Pandas

This is the kind of thing SQL can do too, but it's a lot more natural in Pandas once the
data is local: per-user aggregates, a pivot by device, and a derived engagement column.

In [4]:
per_user = (
    df.groupby(["user_id", "name", "plan"])
    .agg(total_focus_minutes=("focus_minutes", "sum"),
         avg_focus_minutes=("focus_minutes", "mean"),
         sessions=("focus_minutes", "count"))
    .reset_index()
    .sort_values("total_focus_minutes", ascending=False)
)
per_user

,user_id,name,plan,total_focus_minutes,avg_focus_minutes,sessions
2,3,Chloe,premium,245,81.666667,3
0,1,Alice,premium,135,45.000000,3
3,4,Dev,free,55,27.500000,2
1,2,Bob,free,45,22.500000,2


In [5]:
device_pivot = df.pivot_table(
    index="name", columns="device", values="focus_minutes", aggfunc="sum", fill_value=0
)
device_pivot

device,desktop,mobile
name,,
Alice,75,60
Bob,0,45
Chloe,165,80
Dev,40,15


In [6]:
# Which plan tier focuses more on average? SQL could group by plan directly, but combining
# it with the per-user aggregate above (already in Pandas) is simpler done here
plan_summary = per_user.groupby("plan")["avg_focus_minutes"].mean().round(1)
plan_summary

plan
free       25.0
premium    63.3
Name: avg_focus_minutes, dtype: float64

In [7]:
conn.close()
print("Connection closed.")

Connection closed.


## Insights

- SQL did the heavy lifting that's cheap for the database, joining `users` to
  `focus_sessions` and filtering, before any of it touched Python memory.
- Pandas took over for the parts that are awkward in plain SQL: multi-metric
  aggregation, a wide pivot by device, and reusing an already-computed DataFrame
  (`per_user`) for a second-level summary instead of re-querying the database.
- In this small sample, Chloe (premium) has the highest total and average focus
  minutes, and premium users average noticeably higher focus time than free users,
  the kind of pattern that's easy to spot once the join result lands in a DataFrame.